In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Download NLTK assets (only first time)
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Text Cleaning

In [ ]:
import pandas as pd

file_path = '/content/drive/MyDrive/Research_Project/Tickets_Data_20k.xlsx'
df = pd.read_excel(file_path)

df.head()

,Subject,Body,Answer,Type,Queue,Priority,Language,Tags
0,Unexpected crash of the data analytics platform,The data analysis platform crashed unexpectedl...,I will help you resolve the issue by restartin...,Incident,General Inquiry,low,en,"Crash,Technical,Bug,Hardware,Resolution,Outage..."
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,"Feedback,Sales,IT,Tech Support"
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,"Technical,Product,Guidance,Documentation,Perfo..."
3,Hospital service problem,Media data was locked due to unauthorized acce...,Returning to your email complaint about the at...,Incident,Customer Service,high,en,"Security,Breach,Login,Maintenance,Incident,Res..."
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,"Security,Customer,Compliance,Breach,Documentat..."


In [ ]:
df.shape

(48587, 8)

In [ ]:
df['Subject'] = df['Subject'].fillna('')   # replace missing subject
df['text'] = df['Subject'] + " " + df['Body']

In [ ]:
df = df[['text', 'Type', 'Tags', 'Queue', 'Priority', 'Answer']]
df = df.dropna()

In [ ]:
print("After Selecting Columns:", df.shape)
df.head()

After Selecting Columns: (48574, 6)


,text,Type,Tags,Queue,Priority,Answer
0,Unexpected crash of the data analytics platfor...,Incident,"Crash,Technical,Bug,Hardware,Resolution,Outage...",General Inquiry,low,I will help you resolve the issue by restartin...
1,Customer Support Inquiry Seeking information o...,Request,"Feedback,Sales,IT,Tech Support",Customer Service,medium,We offer a variety of digital strategies and s...
2,Data Analytics for Investment I am contacting ...,Request,"Technical,Product,Guidance,Documentation,Perfo...",Customer Service,medium,I am here to assist you with data analytics to...
3,Hospital service problem Media data was locked...,Incident,"Security,Breach,Login,Maintenance,Incident,Res...",Customer Service,high,Returning to your email complaint about the at...
4,"Security Dear Customer Support, I am reaching ...",Request,"Security,Customer,Compliance,Breach,Documentat...",Customer Service,medium,"Dear [name], we take the security of medical d..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 48574 entries, 0 to 48586
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   text      48574 non-null  object
 1   Type      48574 non-null  object
 2   Tags      48574 non-null  object
 3   Queue     48574 non-null  object
 4   Priority  48574 non-null  object
 5   Answer    48574 non-null  object
dtypes: object(6)
memory usage: 2.6+ MB


In [ ]:
text_cols = df.select_dtypes(include=['object']).columns

avg_words = df[text_cols].apply(lambda x: x.str.split().apply(len)).mean()
print("Average words per text column:")
print(avg_words)

Average words per text column:
text        59.913102
Type         1.000000
Tags         1.618520
Queue        2.259872
Priority     1.000000
Answer      58.465146
dtype: float64


In [ ]:
stop_words = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

In [ ]:
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Initialize
stop_words = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

def clean_text(text):
    # Convert to string
    text = str(text)
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove placeholders/tags from ticket data
    text = re.sub(
        r'\b(name|customer_name|tel_num|email_id|ticket_id|userid|phone|address)\b',
        '',
        text
    )
    # Remove numbers (optional)
    text = re.sub(r'\d+', '', text)
    # Remove punctuation and special characters
    text = re.sub(
        '[' + re.escape(string.punctuation) + ']',
        ' ',
        text
    )
    # Remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    words = text.split()
    # Remove stopwords + lemmatization
    words = [
        lemma.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    # Remove duplicate words while preserving order
    words = list(dict.fromkeys(words))

    return " ".join(words)

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df.head(10)

,text,Type,Tags,Queue,Priority,Answer,clean_text
0,Unexpected crash of the data analytics platfor...,Incident,"Crash,Technical,Bug,Hardware,Resolution,Outage...",General Inquiry,low,I will help you resolve the issue by restartin...,unexpected crash data analytics platform analy...
1,Customer Support Inquiry Seeking information o...,Request,"Feedback,Sales,IT,Tech Support",Customer Service,medium,We offer a variety of digital strategies and s...,customer support inquiry seeking information d...
2,Data Analytics for Investment I am contacting ...,Request,"Technical,Product,Guidance,Documentation,Perfo...",Customer Service,medium,I am here to assist you with data analytics to...,data analytics investment contacting request i...
3,Hospital service problem Media data was locked...,Incident,"Security,Breach,Login,Maintenance,Incident,Res...",Customer Service,high,Returning to your email complaint about the at...,hospital service problem medium data locked du...
4,"Security Dear Customer Support, I am reaching ...",Request,"Security,Customer,Compliance,Breach,Documentat...",Customer Service,medium,"Dear [name], we take the security of medical d...",security dear customer support reaching inquir...
5,Concerns About Securing Medical Data on 2-in-1...,Request,"Security,Product,Feature,IT,Tech Support,",Technical Support,medium,Thank you for your concern regarding securing ...,concern securing medical data convertible lapt...
6,Advice for backing up medical data in HubSpot ...,Request,"Backup,Security,IT,Tech Support",Technical Support,medium,We recommend backing up medical data in HubSpo...,advice backing medical data hubspot crm postgr...
7,Problem with Integration The integration stopp...,Problem,"Technical,Integration,Bug,Resolution,Outage,Do...",IT Support,high,I will look into the problem and call you at <...,problem integration stopped working unexpected...
8,"Assistance Request Dear Customer Support, I am...",Problem,"Technical,Bug,Security,Maintenance,Documentati...",Product Support,high,I have received your report about the data blo...,assistance request dear customer support writi...
9,Support Request The latest data analysis repor...,Problem,"Bug,Performance,IT,Tech Support",Product Support,high,Please provide additional details for further ...,support request latest data analysis report in...


#Preparing Data

In [ ]:
!pip install transformers datasets scikit-learn pandas

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import numpy as np

#Type Classification

In [ ]:
df.columns

Index(['text', 'Type', 'Tags', 'Queue', 'Priority', 'Answer', 'clean_text'], dtype='object')

In [ ]:
#Encoding labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Type'])

In [ ]:
#Train-test split
train_df, test_df = train_test_split(df, test_size=0.40, random_state=42)

#Converting to HuggingFace Dataset

In [ ]:
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']])

#Tokenization

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize(example):
    return tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64   # faster
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/29144 [00:00<?, ? examples/s]

Map:   0%|          | 0/19430 [00:00<?, ? examples/s]

#Loading Model

In [ ]:
type_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(label_encoder.classes_)
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#Training Args

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    logging_dir='./logs'
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

In [ ]:
trainer = Trainer(
    model=type_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
#train Model
trainer.train()

Step,Training Loss
500,0.615135
1000,0.471454
1500,0.458011
2000,0.417735
2500,0.377776
3000,0.394415
3500,0.357233
4000,0.338918
4500,0.292324
5000,0.290610


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=18220, training_loss=0.19354058290810275, metrics={'train_runtime': 2425.3448, 'train_samples_per_second': 120.164, 'train_steps_per_second': 7.512, 'total_flos': 4825959455170560.0, 'train_loss': 0.19354058290810275, 'epoch': 10.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 0.8558452725410461,
 'eval_accuracy': 0.8641276376737005,
 'eval_f1': 0.8635439153469472,
 'eval_runtime': 37.165,
 'eval_samples_per_second': 522.804,
 'eval_steps_per_second': 32.692,
 'epoch': 10.0}

In [ ]:
import torch

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model to GPU
type_model.to(device)

def predict(text):
    type_model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    # 🔥 FIX: Move inputs to same device as model
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = type_model(**inputs)

    preds = outputs.logits.argmax(dim=1).cpu().item()

    return label_encoder.inverse_transform([preds])[0]


# Test
print(predict("User unable to login to VPN"))

Problem


In [ ]:
#Saving Model
type_model.save_pretrained("type_model")
tokenizer.save_pretrained("type_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('type_model/tokenizer_config.json', 'type_model/tokenizer.json')

In [ ]:
#Saving Label Encoder
import pickle
with open("type_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

#Queue Prediction

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import pickle

In [ ]:
#Encode Labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Queue'])

print("Labels:", sorted(df['label'].unique()))
print("Num classes:", len(label_encoder.classes_))

Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Num classes: 10


In [ ]:
#Train-Test Split
train_df, test_df = train_test_split(
    df[['clean_text', 'label']],
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

In [ ]:
#Convert to HuggingFace Dataset
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
#Tokenization
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    enc = tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(example["label"])
    }

train_ds = train_ds.map(tokenize)
test_ds = test_ds.map(tokenize)

Map:   0%|          | 0/43716 [00:00<?, ? examples/s]

Map:   0%|          | 0/4858 [00:00<?, ? examples/s]

In [ ]:
#Removing Extra Columns
train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ['input_ids','attention_mask','labels']])
test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ['input_ids','attention_mask','labels']])

In [ ]:
#Setting Format
train_ds.set_format(type='torch')
test_ds.set_format(type='torch')

In [ ]:
#Loading Model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#Debugging Model
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
#Training
training_args = TrainingArguments(
    output_dir="./queue_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,   # keep small for speed
    logging_dir="./logs",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
500,1.541755
1000,1.475543
1500,1.443494
2000,1.452241
2500,1.476770
3000,1.419293
3500,1.307989
4000,1.330748
4500,1.289878
5000,1.270585


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13665, training_loss=1.0720101494340726, metrics={'train_runtime': 1468.1236, 'train_samples_per_second': 148.884, 'train_steps_per_second': 9.308, 'total_flos': 3619856866176000.0, 'train_loss': 1.0720101494340726, 'epoch': 5.0})

In [ ]:
#Saving Model
model.save_pretrained("queue_model")
tokenizer.save_pretrained("queue_model")

with open("queue_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Loading Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("queue_model")
model = DistilBertForSequenceClassification.from_pretrained("queue_model").to(device)

with open("queue_label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
#Prediction Function
def predict_queue(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()

    return label_encoder.inverse_transform(pred)[0]

In [ ]:
print(predict_queue("User unable to login after password reset"))

it support


#Priority Prediction

In [ ]:
!pip install transformers datasets scikit-learn -q

In [ ]:
import pandas as pd
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
import pickle

In [ ]:
#Encoding Labels
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Priority'])

print("Labels:", sorted(df['label'].unique()))
print("Num classes:", len(label_encoder.classes_))

Labels: [np.int64(0), np.int64(1), np.int64(2)]
Num classes: 3


In [ ]:
#Train-Test Split
train_df, test_df = train_test_split(
    df[['clean_text', 'label']],
    test_size=0.1,
    stratify=df['label'],
    random_state=42
)

In [ ]:
#Converting to HuggingFace Dataet
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [ ]:
#Tokenization
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    enc = tokenizer(
        example['clean_text'],
        truncation=True,
        padding='max_length',
        max_length=64
    )

    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": int(example["label"])
    }

train_ds = train_ds.map(tokenize)
test_ds = test_ds.map(tokenize)

Map:   0%|          | 0/43716 [00:00<?, ? examples/s]

Map:   0%|          | 0/4858 [00:00<?, ? examples/s]

In [ ]:
#Remove Extra Columns
train_ds = train_ds.remove_columns([col for col in train_ds.column_names if col not in ['input_ids','attention_mask','labels']])
test_ds = test_ds.remove_columns([col for col in test_ds.column_names if col not in ['input_ids','attention_mask','labels']])

In [ ]:
#Set Format
train_ds.set_format(type='torch')
test_ds.set_format(type='torch')

In [ ]:
#Load Model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#Debug Mode
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
#Training
training_args = TrainingArguments(
    output_dir="./priority_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=8,
    logging_dir="./logs",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
500,1.057245
1000,1.036837
1500,1.018006
2000,1.012886
2500,0.996888
3000,0.964950
3500,0.932430
4000,0.930049
4500,0.908643
5000,0.904773


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=21864, training_loss=0.5828958546945214, metrics={'train_runtime': 2356.2067, 'train_samples_per_second': 148.428, 'train_steps_per_second': 9.279, 'total_flos': 5791048072925184.0, 'train_loss': 0.5828958546945214, 'epoch': 8.0})

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='weighted')
    }

In [ ]:
trainer.evaluate()

{'eval_loss': 1.0176414251327515,
 'eval_runtime': 12.5179,
 'eval_samples_per_second': 388.084,
 'eval_steps_per_second': 48.57,
 'epoch': 8.0}

In [ ]:
#Saving Model
model.save_pretrained("priority_model")
tokenizer.save_pretrained("priority_model")

with open("priority_label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Loading Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

tokenizer = DistilBertTokenizer.from_pretrained("priority_model")
model = DistilBertForSequenceClassification.from_pretrained("priority_model").to(device)

with open("priority_label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
#Prediction Function
def predict_priority(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()

    return label_encoder.inverse_transform(pred)[0]

In [ ]:
print(predict_priority("User unable to login after password reset"))

high
